# Executive End-To-End Evidence

Presents the final claim boundary, dataset scale, forecast gate, planning outputs and operational lifecycle evidence.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / "outputs").exists():
    ROOT = Path("Ai miroservices/modeling/project_operational_baseline").resolve()
OUT = ROOT / "outputs"
EVAL = OUT / "evaluator"
sns.set_theme(style="whitegrid")

In [2]:
manifest=json.loads((OUT/'manifest.json').read_text())
evidence=json.loads((OUT/'forecast_evidence_summary.json').read_text())
pd.DataFrame([{'dataset':manifest['dataset_version'],'hash':manifest['dataset_hash'],'materials':manifest['row_counts']['materials'],'locations':manifest['row_counts']['locations'],'orders':manifest['row_counts']['orders'],'tasks':manifest['row_counts']['tasks'],'champion':evidence['champion'],'test_wape':evidence['test_metrics']['WAPE'],'coverage':evidence['interval_empirical_coverage'],'promotion_status':evidence['promotion_status']}]).T

,0
dataset,PROJECT_OPERATIONAL_BASELINE_V3
hash,ba12a1d46e221a5feefa890e10976ef0df76493dab75e1...
materials,80
locations,606
orders,5000
tasks,30000
champion,EXTRA_TREES_RESPONSIVE
test_wape,0.127608
coverage,0.891493
promotion_status,PENDING_MANAGER_APPROVAL


In [3]:
for key,value in evidence['promotion_gate'].items(): print(('PASS' if value else 'FAIL'),key)

PASS beats_seasonal_naive_by_5pct
PASS wape_at_most_15pct
PASS absolute_bias_at_most_5pct
PASS empirical_90_interval_coverage_85_to_95pct
PASS critical_class_wape_at_most_25pct


In [4]:
evaluator=json.loads((EVAL/'evaluator_run_summary.json').read_text())
claims=pd.read_csv(EVAL/'claim_evidence_matrix.csv')
assumptions=pd.read_csv(EVAL/'assumption_registry.csv')
display(pd.DataFrame([evaluator]).T)
display(claims)
display(assumptions[['assumption','status','evidence','action']])

,0
data_tier,GENERATED_OPERATIONAL_BASELINE
history_months,24
forecast_horizon_months,12
series_total,96
rm_pm_series,80
finished_good_series,16
selection_origins,"[2023-07-01, 2023-08-01, 2023-09-01, 2023-10-0..."
untouched_test_window,"[2025-01-01, 2025-12-01]"
neural_seeds,"[17, 42, 101, 303, 707]"
baseline_champion,EXTRA_TREES


,claim,evidence_required,synthetic_status,production_status
0,The implementation is leakage-safe and reprodu...,"Feature invariance tests, train-only normaliza...",SUPPORTED,UNVERIFIED
1,Cyclic and spectral features recover temporal ...,AR(1) spectral tests and feature-group ablations.,TESTABLE,UNVERIFIED
2,The neural network is superior to simpler cand...,"Locked selection ranking, HAC/DM test, block C...",RESULT_DEPENDENT,UNVERIFIED
3,Forecasts are suitable for operational invento...,"Real costs, service targets, lead-time outcome...",PROHIBITED,UNVERIFIED


,assumption,status,evidence,action
0,Temporal ordering and no random K-fold,SUPPORTED,Expanding-origin H1-H12 protocol with a locked...,Keep all preprocessing inside each origin.
1,Annual seasonality is present,SUPPORTED,33.3% of series pass AR(1)-red-noise annual sp...,Retain cyclic and spectral ablations; do not f...
2,Series are stationary,NOT_REQUIRED,69.8% pass joint ADF/KPSS evidence.,Trees and neural models do not require station...
3,Residuals are centered,REJECTED,HAC intercept test on monthly aggregate residu...,Keep bias in the selection score and monitor b...
4,Residuals are serially uncorrelated,SUPPORTED,Ljung-Box test on monthly aggregate residuals.,Use HAC/block inference regardless of outcome.
5,Residuals are Gaussian,NOT_REQUIRED,Jarque-Bera is reported but Gaussian intervals...,Use empirical/quantile intervals and block-boo...
6,Residual variance is constant,REJECTED,Breusch-Pagan and scale-error tests.,Use scale-normalized intervals and slice metrics.
7,P10-P90 intervals are calibrated,SUPPORTED,Untouched-test block-bootstrap coverage interval.,Recalibrate on pre-test residuals if rejected.
8,Generated sample represents the operational po...,UNVERIFIED,All current histories are controlled/generated...,"Require representative real issue history, pop..."


**Supported conclusion:** OptiWMS can execute a reproducible, statistically controlled warehouse lifecycle using the generated operational baseline. **Unsupported conclusion:** these results establish performance on an unseen external customer warehouse.